# Sconto v2 · Consulta operativa

Este cuaderno reproduce las decisiones de la API sobre la misma copia SQLite de intradía, catálogo y hora de evaluación. Trabaja sobre copias transaccionales locales. El análisis histórico y el modelo retrospectivo siguen en el cuaderno 04.

Proporciona las rutas de las dos bases y una fecha ISO 8601 con huso. Puedes limitar por identificador de producto. Para reproducir la API pública deja `PRIVADAS = False`. La revisión privada requiere autorización de acceso a esa entrada. Mantén los archivos operativos en custodia privada.

La comparación actual ordena totales de ofertas online verificadas y vigentes con mismo GTIN, condición, moneda y contexto de entrega. La estimación de ahorro requiere historia previa del mismo anuncio. Capturas posteriores a la base no amplían su cobertura. El registro guarda versiones, hora, hashes de las copias, estado y motivos de abstención.

In [ ]:
from pathlib import Path
import os

INTRADIA = Path(os.environ.get('SCONTO_OP_INTRADIA', 'intradia.sqlite3'))
CATALOGO = Path(os.environ.get('SCONTO_OP_CATALOGO', 'catalogo.sqlite'))
MOMENTO = os.environ.get('SCONTO_OP_MOMENTO', '2026-11-15T17:00:00-06:00')
PRODUCTOS = [p.strip() for p in os.environ.get('SCONTO_OP_PRODUCTOS', '').split(',') if p.strip()]
PRIVADAS = os.environ.get('SCONTO_OP_PRIVADAS') == '1'
SALIDA = Path(os.environ.get('SCONTO_OP_SALIDA', 'decision_operativa_v2.json'))

In [ ]:
import base64, hashlib, io, json, re, sys, zipfile
from IPython.display import JSON, display

bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQCZ24jMTgAAAFAAAAAWAAAAc3JjL3Njb250by9fX2luaXRfXy5weVNSUgpOzs8ryVcoSk3MSVXIV0hLzCnOtwJyk/NzUpOTMw9vzlPILc0pyUwrTc0rSVVISVUoAEpm5heDmCWpyXn5Ofnph9cm6ikpKXEBAFBLAwQUAAAACAAAACEAduobeOQLAADLJQAAGQAAAHNyYy9zY29udG8vY29uc3VsdGFfdjIucHm9Wltv3MYVfjfg/zBRCpBMNkSSIi0gYGvY0jpVa0uuZDcp1AUxS85KjLic9ZDUJY5/UB/ylLe++o/1O3Mhh5eV7DxUMGDOcObMuX7nnOHu7e0dyLJqipqzs3+8yGvB7liKmVo1aZp/+K1kSmyVzJo0XxWCZfQvzatclqJiW644e/rqCHtKWYuVlFfx3t7e40drJTcsSdZN3SiRJCzfbKWqGS+xjNfYXD1+ZFfhsFrc1kW+cqvSQlZ5eWHfZ7wWdb4R7i2NHz+yg58qWdp1W15fekReYdguU92O6m0BIf/YjpsyT2UmQJVbQo0qQCeGbFV7KOaqLTa2XFfEtow7XSTX37rFYSpXQkFynlzmVS1VnvIZ5NyQulQi13jJq4SndcMLUc2YuOZFgzeO2uzxI/bgXynVhhf5z9h3UeflDEKCsCjTnCdZzlXOoz6veVkrnvE7x+bB4csfSZ7Hj14fLY4Pn56xOXsX3NzcxJWA7DEYjje3wYwFwzGtKfJrobZSFt78aO69oZ+JNSuEUAmUH6qmhjbwNNPus6nmYRTtG4lv8vrSmT+0lgKtshRpHZJF9e4oVqKSxbUIoxh6bFQeRuxLFjzZwJBzJcEK5uavVSNmjFxHNvX8uyhivCJvK/c79dIwVvImWcMaUt1BBe7YU3kzWJcqAd+DU5cpuXAYtCbI6MxvZizpjDJDnNRCbfISLpCnmpvoIYKiLHgqkm2zKuAziDdLdjT/AHUl4H0lO89y6G2dFzxia6kYPbG8NIeLW5E2tQg9S0TLzl6eKCHcUypnI0s7COKfZF6GqaacElkvlGK3W4TB8fO/H0IOQ4XhXxBEccorsZZFBttPOnu+JkTpkYRPrSAvPCOlTS2jI+WEiFbHba3uPHNDzBqwNW/jWS8178VtKrY1+ydCUSyUkmp/pM/nvKiEmf2cPaXlXLHqw2+FZIYJ8q+NUAjBgp55CfyEAABKJa5zvhGAOkaPFVymwiZGBtHhyDXYxj0dh4bfuEovBQBwPmfBZV1vqwBAmllh4lLUhUzjQt4IhTiAHVw4e4sIHGnc1zVp2C5YK35B7OlNSsTrpig2vE4RcUH4ZL+6avJs/u/sy+hJMHN73jZC3eHEShM6Bgpas6QFryrm8orVo7ZVAgPWSRJWolgDBBBycOUU1i3khZwTiRkBvKwS5Jy1SOVc6xxHqvwa4FWZceTZhkjFmhIFL/0/eOfI47WGEDeOyMnalwJktQyD3T1uQKI3Hqx1TGKZeyR1OOkJ/azgk+jnWb5Fy0464oSvOCnDD1n/AJOnwZ895u2MFfkGaDZrXyUwo1EzvJMX+jnqA6IpBuY9BHgbxSZevGjN1z5V3w32+24G2KmVdByDcuDvmzNyqXNvatltJ6tMUiMq7OnxIbMwtJdTyRJ6iIznzQpO8csvWBnQf94R0Yw9idhf2Nd7Gr4Si4pa9Ehj1Dfzb4L+wS37bqGntc7Fzpc9DfU90NORDjRtA/YZZKlzUWY8WSP/pzwYiOyR7zuGezFj6yAIxkB6tnixOHjtCz5jRi066sQFahMkY+SdpMhLwaHSM8vU89OTlyycLkSGZLFDEs6ROUHg/uMSvt2aY+zE9Bn6fCdfMla5+4vYD39dnC7YO+MX79nJ6eHilD37V3uy72wvjl4evWZP+mSgOqSnL4x9XchE0diOfSzoG7BV4Q7bId/WMLENaBjw0y0W2JOTVYPT1nkZ9PQ4G3rR7zMmnfjy6Dg8OHn6YnF2sAipxk7AvUKJhNK2RFFhKl56gUP/EG+4wnGRDbUZJXg87y5iH6aYiSpV+ZamAxRuEMQo4R5fsWU1+9vJ0TGTq0qoa27JmnK4QqGlE/Cbs6Pj70O5xgaCgmmaxqvMjvmE5r8/PXnzitzMU9z/0zs/Coi1j9KgRZBPQB2qFgnuvBRkdRz2nOXP3w1UCH7MXjqMl3fG583eBLUPNZITVaneE+2PldiGEJCDQkhXtX5Cm4iW/lwbI73c18VsNK6dK9RkIgs7hL0Sd/OCb1YZ15zus9BPkDR1HpizgyXSi5nwuMBsdL5vzLf0s/ZAMSZ1m8mJyqBfaEB9oVl6HhhfDZa6TrSW3fASPWYwLv3cJi/6gmV8Ieow8LoPXdxRb6FNOVFpOzLmDYnoS3YNhqDphKoWKxe/lKon1lSOnKghrPT9As156K6kGFhso/Op8oScMzNACBvUEMQjSnETpmZELoWa5J4mHOD97Pk7LQs6UGoxfwZjIXXUUXyHbvl9MJvwKc3v+dfLLkK8qrNTW9fI95zBKs9IMFAhxThNk0OM/OGzkT98inLbhDXKV1a7Dkjh8yK95EmabW4Jsc0InY9W+wbEM054Z8YGoUWW8BoFrWwUHKvZ0uWOnppUvt5LFyRJLahp4iiDs7zaovhY5UWecQr1QTrpEzKZopcczLVIzq0LOGnmT3R5adi2A0++Z4vXPywWx8y88CDbwLWzgaMWLGetXQxJmiGLnQeedw4mKdEs+350gX5fyWpQaI5gdFiEdErBTlIL+kWeVRq24q3chsFQb8GwMadmqVszDRXkhi1c0KCHVlPQLss6Lxsx9EcrpkN8VJjavImtC5CMNOtdaHisTQTe6GosbI/oSHh2mJMfxnR5lldyTShfhxPGij7mos5Z8gGaxtYzdinv5juAhQiEfYR1CdngRC/nuc5vhLi/K69rcLF7erBuyHcLXRnWd9DP2eIWYSoUa0oO1ip0mKlQdB9JV8opRzjDFHmZFk0l7cXItV6W2RsTfRddSHhC7DdetBGHhV2ozXr9JQ0tEtKLAslBK59GECQTmVT6mdzCrmqdaToJuGynDyI0oycb1e1cUnOAhb4YDDys09epQ7Tbccxa1OmlXpA0ddqS7gEgzfYhMIg+Hhds5thZn310xJrNdGkwEanvrvZNOXS11Bxd6W5bW+69KZRcudSrRqKpI5DWvLRCxY7dOpi2grt5wuxgOUQZBwrOsb3EO5Vyp/lxl/aal+FNfkunI21pzoeB49PsPiCYi0NNe+KrwiT5nby2cGqLyxFM2mWTVar9fEHGsV8upkBnBDaW8s6PH35vMMIfR68HeVZ4vhvzBrWRtwJwx1Vd0WeGMHh1evJ8cXDy1bPn3/7pK+QvylnjNn8QAgCywtYwbP3hV/RNEpvA2HUjimvBEJQs+/ArZ3esatJGVVSjVPD3vNSwqRhJRKBWwFcwscmp5IyHrtnV+pOlF/3Z8iuLuwIsi/tFVxa7siuLuzl9Ea/bYmQYqn53p7EsbgpkId5ewBASmeLOxyYk0diB6e5riN3HAMLMuqkKb/raw/byuqAzotk+X7Z9/kQ/bi4HbIqSw5sAVwLGfnNpKjwZ77wKGJ/SNvl9e7DDxdnBbGAwY6D7Wv8pD5+4nNpRZfU+BSl5M25+Z71KfY41FjWpGvUnNwIKLvkImd0fYTsWErp7tcXAOJPi9CqFF9xVc5TxESm2yjRjBBvf0pBhGZUP2wYuPGOIpFX+4T+AmgaBLKl++PBf7btYtiXYpJSDJb/FI3R6sMUpPZOV8X1tTtlGHN2hlVZ6fYNFH+s2cjoIXj79cbCY39JidnDy5vh1+IWeIxBVcmUbl8lr22nqw9uAQfjs6IfKB6Klf9THhM6QjePDMbv6brC+2wroY/Jq0O85QBLdhvkQML6cKGMzN+C0vbnzrOobdeZs6F3W+TbXUdzbuyuG743f9vvlRMWERyAzzJ9n0NyosXUJzigSpUG77N5bILMJeGHGHlDY886HpeRS/yTATULj+9rpENjpFVWe/ELWVOPud4/vdWe4k2DkErl+H5saOBxDkssmc0+0tlwHLrlyvffelfDL6TBoNehv6mp9oipUvtZQg0XgHfFYI7palZn7moCr9FKD7Y5LopY8pc11jm4j43NLQV/ADvvoboPjIOPd5dsuaWCGpIbWi+4YOR/Rnl4X7Pzsfh+fKVcXgPKOTOVdEbZ3WTu1T7/zoW85YGiKT/99EA21vpsy/fxlPvg5TDiiT7P0LeGjySJuN1LXuy0tsuNKysKnri95JxbuSpOT9yjTFHrf1jt2o96PBGwoPX70P1BLAwQUAAAACAAAACEARIb7ytMEAADjCwAAIwAAAHNyYy9zY29udG8vY3VhZGVybm9fb3BlcmF0aXZvX3YyLnB5jVbbbtw2EH3fr5j6hWIjq4WLtEAAoWhjp9gisVPbCVAYhjArcb1sJVEmqa2df+pTPyE/1qFI6uJ1igqGlyJnzgzP3HR0dHQpOq2qvhRQI5SqNX1tEVQnNFq5RzBqowUddBINWI2twbKUqsVaGKgEXP32VlqRHR0drbZaNVAU2972WhQFyKZT2gK2rbIERtgrL0NmrHiwtdxEmbJWRrZ3/rhCK6xsRDyM7ym4/59UK1bhZIdmRyherUO7m0G+p9coZ+5rcvI7L2dF021lPcJfC/eL+vFUalFapR+Dm8b5qbJISrE/iSqvw9ZCrhKlNHRLYWaSH88ur9YX5ynduemQrBRqK7RFU2Bpe0fiAkO2RHGFj1F9Hd6vyC2xWq0qsYViCIYuTIud2SmbCC+UUjiMla3ir1ZAj9LyTlKcIB+4iGI808Koei8SY7UsbX6te8Ez8qjXMuHwAtiPjapErhUbcP6SdhcDlAQmHSktkZVEIymQtofiQJmy7ckcRexLetHVQXi4kHfaPV4322D5Z98lwyGfPAmaGaVomzC9YQGCVFpZqQlGC8rDNuZI5kJeVPKO1JNROgVmdnjy8nvGs514CMc8MK1DbUidbNCIwkdHEtMlWqzVHek3qiFvafF1Cl7aKpMn3L1R/VRo8jdYGxGCQnXywSCIGhppGkUmavUHiIeu/vx3KR1Oq3SDtfyEpfz8TwuP0Ck6s7JEV257SrGNrGWFFdz3Q9H+9H49lJ+Dxx1lMgU8lkzmkksatXWgNgnOejbl1otnvS3VdmsEXRykgXNK4RmLKI2Aj1j34kxrpRN2VoNsjUWi0Hkk9nQWnNXivpeC+sWuNwocuJaKzYJ3WGvJED7r9+vJbom6ExZj8kYBPgq4uI4RIbGDsjgImQf8BljczEJWMj4z6wM7whDyTDMeB03mSIx7jjvqdAN/FGBiza2W/o6yz/g75dSBE3xuZ3Lt0MjYwPOxRyWLHpKEy/BnrMxSNi4OiEFN0O6WY66PEpOLqKc4umeUJd0bqkO8YXGrkBW7BUpPcPuUWOMVHMUJY2yB5J6rs7dnr6/hdH11vT6nhcpmYPDm8uIdhB4L6kD514v1OaiNEXqPpW/WYw618OFqff4LJFQNQhMYh4vL07NL+Pn3pZEFKrnIb2dNZ3C+UoYIdn2MMs9XyMYNTLr/bQrf0t/U7ejuEdvd31DfF1XiynHkjfMlofF++cRW2Bp1UvjhZeoLnC9052Gi2VwN0Qzarw74cp1Str1YHPiLwYscTN8kXveGlWpDi15j0Wmxl8hub5jpt7KUrp2HKHthd89gcuncjKon6HG6OlTq0cQwIX6VAyNTlWhLiRTImbn/Y20KVoYdjZMqqWggJrNI5xOdASEPv+kBVc88Yei7RAvzPv/id0ASXeT86QgbvFryZKj5N5gz/91wXPZYCd2q4/jhpo73JywlgLsa8/EjhOa6ytkow9JnyB+cpZGQ++GAJn50JXHhBgbPpqnClzCzEZWz0EiGRhmbiu9brOs3NU21J05Y8m7khlp+ESOQUz4vJP3gHrvXWMf5YjKkB3Kx7+WLjrzEbvtGaBX9yF0i1vS54XtX2GW3fNG4pmzi/wVGfb8tdtJQN5axVPLYKp6ITuWQT8v0+d6az1rPKMFX/wJQSwMEFAAAAAgAAAAhADTpU6pTCwAAYSIAABsAAABzcmMvc2NvbnRvL2RlY2lzaW9uZXNfdjIucHmdWmuO20YS/j+n6HixoJhwtJqxs1kYoYGJMwkGiCeB4wSLFQSih2x5CFNshQ/FTpDD5AD7K0fwxfar6gebpDROVsBYVD+qquvxVXXRjx49eqleV1IUSuS6bvuqk0LXVVkr0eq7Rol9o/JSt6LTnaxUi1W7vWzkHT2/E+pQFqrOSykO5WtVd2r56NGjs22jdyLLtn3XNyrLRLnb66YTsq5BpCvB58ysyXVVqZxH3KJCbSWkKMq8M2sK2amu3Cm/AL8TP5oI+rdQkNs8/qJrZTbuZHfvNpXttqzLzs60JEXblbnnChKlrBPxUy/rrsTZzs7Ofrx++f3Nt7ciFZFRwvnh8nzf6EPZQmBZRWc/vPji5dU32XfPX2HRp2cvbm6zL2+uvs++uPr+GiNPzq5efPfNzasfvuQRnsPw5cXZjzdfX98+v7nC0Mvr56+uMOqPsbjXfdOmlzFEgDJEVtaQF6pdHGSlm/jpmcCna96ZB/q4FSDj9LKkg5at3uoGerBbRbmFJszq3NJLoA3MqKpVggcGqqPFjseg+hgGLTzvZd/lerttVbcAn1bA1OIWthikpE+j4BL1sEm2zmYL90CEYt6k3uZq34nFj7Lq1XXTkLDfHlSzrfTP/DMeiO9l254FHIi1UyBMe1p3dj0d6v/XGS3q3u2V39MyQbOCRQnOMxxnJkUo9U7XnR7J7ZQ3kwbG6RIBtcguNlYh7c/X3WldWavZeHAC05g5yTOxchJ06i0k0FvVUHDlEqFiJTFLU2Hmlq9hdDP9sJysNWJFgpjBJcbK/SL23g4D7GTWqqZUnvW2rORYB7KqFkflY/IzsUSaMpGppMEHRjck4JxiEcGVVZOVRZSICJZQhaQnQCQzzXLddjryUjcKqxkHydka+JuFzUQ48T6G+mWrEM0lJuwPGCER9/pdSoa3JwSAPpdV3gOUc4dGFosHDK63JXy00O1SPJcF4Pv9fyUgRNXkc4J0qMVetZrh2ECEYQuOsBuHRCBOnARDWGK0A8HcWjxyJNBQENzGwT3m1PrncRgvaWphqFEcMTMiwJuhcRIn+GnJu592/TNaFsSKLMF0iKJF9JXK72VLOQxKG2yB/Yf3v1dlIdvIyICjtASSQ45ZVEgEZnLL8lS0zWW9gemWWDhteFeKeDgzq8kd3HK2lAicLmKPyXg8G+wXxSHcLgyXUMEGYlkNn6dWCnqAjDRlB0hxU4c2854/cLPclrkEpTrigIiKEnJ3MpqspIzeQWnFIKeMGNNeNb36AJuibPe6Lu9KUnthGak6w+HzN4bT6RC30wx8rKk4HqcPst+az7xZyv1e1YVdN6CSIgOvTT53URizbV0lA9URneWBPKhdxBvejFhrTKylQxWwsCQTUadPElQJ3b0u0qis86pvy4OKOCgqVbuFsXiWinEdwCGyJmNuxMficYhi7IAOadIAtQb4QWQa+BlNW0SKk5k1xgg12jQFLxyKvLlNSX56OEIOgFiVXV+YhYudfGtWinOAjN1FUf6uJUVwePF5V3NSxotT87Uc0mzMoJTibzyYq6aB59GEcfA5SWNlmVprextAFOcLPgMnYn/5aeqtvF5tMPJZOHK58UkowPO2R9SUVNp6Xx1mx0lpEULPJIaPOn1AiBcNv63FjApOBB/t4KJjus2YFXBE21AaTGiv3YLNzFn/HP2RUzzAZryOuc0L4hlLTu0GAyaM38QmkN+YFG2tT3kZpuWvzz4NQdhRDGWilRtC0HDQUZpNEEHvFbm+g9n6Rmb3yBm6AZgesaTJ7JOEfv0WmsglpSeaJShSB/gUFRsEJVq0pdji5LIGqZ96lE6ddgnN8BAy73pZ+YROdEK+gC0mDf38+ttogkdmsOO1NDh4+uf9fohFG9bE23hHUFZENrYnkxgKocui0AkvTsQKVMY49LBD8g5PHTBlNo0cPeHxMdW5bxoqzv41QVNV/iKb7HVX1qPSHFZBQYKUJ75+dXN7/q9E/PDd80RcX92eXzwW2oxePMGNmW7ZqjlI0dd+NJf1+z+gsaBcQzFwvIbnGhom9UknNsU+IgJcLy7x9xh/T8ZhgA20ypTcZSvbvCwX8XS4QILEEYcJRLVlEh+/qxgH2pH34YvuIYs6RppbPKYj7MXfxSVVACuDwxc2ESPzkMCq7neqoQqxQTQ0rSoMs/XT84tNHI8uE0bEX1AmVAscjhXEjD8ZhFzzNvC8WAVMgzuV7V00mfHoNjMxhTxvBwZrftugCpJhxd0pahW8/71WetQFgUnoLHSRlQ1V3/JeoywVBBLv/2hCq8rXTb8HoE9rUPpnqEFtvIOolWpQ/d/ENbyBgCHnJgQk2zc6V5CVwEJRSYPjuWlomACEU49v7yw9NfJiSDL167BmoIEQUp1f8lYvq1kLSXaaykuqGwuK/9onL9rzlYQ5ZjkcRKhnYAitIyd6tAEs6648aGQOKjd+jSzqSmqIeHCKfpuUiGSpsg5SZV7Jg/J31TWJWfQ5yiBUV6gB6CSJn7SF1WYY8WeZDIal1MYz8xZeM9vN2u3wBd1m42U54220Q3PRuvEusHBCOvmMXInw0iS+0gM6wi+zwWta3XQIJS/KEvf8HercQFG2aB32xSixLj+gSCrRSU5Lf9jsK+lEvFHv0kru7gCE+qlY6PWpew8pM9RK4GM7VXNzgfmhTjtNY/Bk1qG7EnBMBWZOx9pMQ5WmM82mg45nzjpRT1Bou5+JEd8Im/Kzc5s2XU8uM6PqP9RGIlAdFKrQDY27ZxrfV7KTXCTTzPALc3Np9R2lGlVksmP6w08iZUR8yERzikXpC4RDm/Hx0kb3UPoDdOiyYBRxOanOGO7IZ9nWfmozxn5WFAAWxFJ/waxURG5so4eRPvLd6DFEHLk4MLJwkTYjQqK2gNCswNc0UWQB8KPQMBtT8+XqhCkKH6mfUnMj+TjhZCEn1eKX6tCrCqAFDGntpR31ARUPsm9No0PetR2RonHTHBoqRfUW4kqfdRzwF5r6F76bzJw5kfLTX2nruBIloGy7Ng91ab6RghmFrxq42IVSBGibaQC8bYoYd52KPbolBw4d81HG+cgaIDOpW1a2HeFoRZP+bNgiyWxzZNzkfKiLYhpM1MHHnkAz58FBSEL/Y9IcLmTBZQGlaBNDrSIoAq6Kf4h/rviCzeSnd0sz+ix8h7CKp2ej1zPmmjBI8ImYvYgYSchd4WNKOUKb3z4cXUxU+HRTuQMlfe4FNB162z4bty8eaKDZUmHIohNfQIXN5v/IdKPo/ZaMBme1210CMctxT2lRvUXe3blUPnLC03S2PSkmQ4GYHdtoSavKETeKfIAetxtH4EaCOe0OsRsheDV4QqNFcIJR9Jzu85Es1G4Qwdsg+tD46c77Ax30A4U1Q7EYspnpp9viKg7Lk+mxB0lxXQYAV8rrbmLnWfvxo7D9eJrDeB8bzCmliT7IZKx62sz85m5q7TxqcU7LaNfbY9Ff/Ps2cpexidZn/bzTpzNBU+sge00E+osNr2NMjpfnLpEPGWiUzU9lbEc9td+JvVVl+7wzyfN4bUaRkOFq33fYawoTGkro6hkEyjjRPUDvOOikBp/K7cSSH6Jm4zu7lzh+hkyaOtQLW5/LRqGuw40/+mS1erpaUaD8x3SaPYp/iFNDL/VT+wZ7Vnl421vtzu723l7m2moutCnu1Su6218gpxkN/ON4G21i9GW/5/dAzugcIIoQFgVOQw28wLjGaGYAZlsdaUpbSf2RfuW71tNp+5RHLSzxHZBg6XRJf6KfmFC9wM0sPFHnCl++NTVthf7mNesPv46Gw5keaPC/BrjFypqcNh6pTRmaxVMzSow2w39LyLbyoE1QD+nkFH/wOT8qwLMjnc8/K4CsgD9hxT74z/8AUEsDBBQAAAAIAAAAIQDxBBgzOBcAAJBYAAAWAAAAc3JjL3Njb250by9pbnRyYWRheS5wecU823bbNrbv/go0L6RShpXTdtaqWiWj2EqqU1vOkeUmrevFBUtQzIYiVJJy7PTMx8wHzNP5hP7Y2Rs3AiR0cU47o5XlkLhubOw7Nvjo0aPhbTpn+SylJM2rgs7/+BclJVtReKRkzjIy41hecTJPaZFycg+lJOMlmRXrOfxH84pBecHK+NGjRweLgi9JkizW1bpgSULS5YoXFbTKeUWrlOflwYEqe/cxXennG1reZOm1fv215Ll+XtLqRj/zUj8VTD+Vv2Vpxb7UrxVbrhZpxiQkM55lbCbmjen1TIMzApjpdcYi+VTxQjeH1dxVAIluOYOlpvm7SFctaU7fMdV8Tis6y2hZslK3N0URWaQsm5uGrEqXzGoFc+vSiOBfQHZF5eNHniv4V7B4C5rXiAtRUd2vACxdPsjvZfG6yKB5DPtXmsmgrFwBjmQLHDvNF1zX/gzvI3g/kNUlrpPHxbqiZk3Hg+ng4ODH4eR8dDYmfRLM6Aq2lz65fRofBroiOTqbjn4eHOk2vEo/0hlg/kmVsnyuWx8dn76Fej1tGAyWQD8z+sUpu0tnPDlKq/ugczAZjH5ORuPpZHA8GkB7hIF8QQK+gv3CUQN8u30q/pOkm9LgYHg+HRyfnUOH3wP+PohIALuX8AUrKopv1xn/bc3onONLzhMYbZlWqXxf0CwTDynSdLFerrDiHwcw6n9fDE8RjCCAFUyGg+mQTAcvToZk9JKMz6Zk+HZ0Pj0nAMY1/RW4Ijwg8FOvSTon0+HbKXk9GZ0OJj+RH4Y/RaLBYs1gKlmJw4wvTk5kDWyar/gWeXUJ+Kx4sq5mviZLepfgAqBJSQCDw1fDiWlCjr4fHv0QOm2e9clhR/a9GI9gpaEEK0IgouaUnYPOt9tQYIaVKJCkwjajoIkjA+lk+HI4GY6PhjVaw7qxAljN1lqnrk2BVDYhagGUoatkScHUeDVdmJEl4vwtSNiNAIcKpnUGPMwToCBeWIMr3NZLiDT0u1Cq2G0PlNpI0xsR1u07n0h1C1bNbtg8odUmXLKyAqZKbqpqpTGmUVoCPqDO16u8oU+//pu9ASB2ElQs1Cq8ZUUJDJ+ALAdRrnDaGEnuTqgGHJ2Lqg4wbGgNqYt34VuKC41uvoDXT2Fhmq9zJL8mZcvaVcHna1jOhupbULZUD6x6ZKBbFrxY0g2SYc7mDsWBKJ8jA+RWGcszOmuAa6FjND4evvWjI9EQExDwqiy0VrEDp9clK26F5GZlogX2Bor2yQDNBG1ydjfI11dDq1vqjgImQdcWhkq+LmYsWa9QPTfqVgWbpYIY0czgBW2wi6JDTzMUEqBXgE1gU1HHoLrSEmPBZjc0Eb24NZssns2Xd34CEs0B5Sfe+Tl5RrpqgiVgfe4lG9k0AaPlBplUjGYPYmrqweZpueJ5ep1m6ZzOffQGO4wWnFdQeOhP1licZe1xRBqbpjjdg+K+RCoZjI+drVVsL8odTGOFAqEjxobf2YRsGNtsnncCPU5nFzM5fCAJM0FOQKby8oih2sgiiO2TaF5JpHBKRD+cwTCR1vCuZH8ICwt3wEzxn2XkFv/8JayyD3nvIuLIgTVSs+0iGul6me2UWGhTjLslLbqRc9Yz/TgavmlMJJuokcyEZHB+cD48GR5Nd6wj0lx0NDgfkjffD8fkdDRWOEZtfDp4a96mzeohsNVAw2CGwhYNKpWtwE8qgBnAs0MDAyvqPjCNp48wzDZ0OTq7GE/Dx6KdbgA+nCw+BuSMxrB8DSo0uqUZOr2AqrJK0c46eDk5O93OJP91Nqp5kFycj8avbC44eDU5u3hNXvy0A83fHqAbcnAwZwuS0BsQUGGHPHlm/Mmesrxg3NwUxjn/EGrnMhaGvBoBnkOxmp5pLIYrq0KOlC7kauPqo/QcSzKGQQgYG7IcRgCIS1YBHKqypxFLCpqCL/ojzdZsiCZxGJxQgkADhL+tU1Ywss7JzbrkhN2tsj/+NUsrHnTsNchZaKnBd9cRpyUXZlElyssVm/WDZToreMmQZ8ugExdshXonDD7vdnvdLirgnwODgXQePgaXuWJlD1dN/kcswUWCAkUFKmJpaYYoB+I5eIpgCokBItBwJe4nLWdp2n9Js5J1YnCh+JyFnU58w+7m6Tswl0Mz+yyjtyzB4ALX+9AGQr06sAQkiH/laS57xTNasgXP5mEnFv4+TGe2jjCAQ4ygZ4ViUN+Flilg+SulIGYXvp94EgDUU8tVAivrqEIIDx1NJmHOMTQTL9ZZBvsxuwmL4JI++XiFf7pPvkmuHgPq5TwdpB85XFwCqy4Z+QzULLoQZWCoR+iFguC4qu0NL6ucLpnVfQ08J4qA+LClpk5VvaJl+YEXc7u6OYFuivEObJPmJMR2Efnqqy9tUBcFfbcE+LEIVloyWshl/lL+8ssvd93uk1/uDhdXgcBgp7OND15KqcDJxeSEfD+dvkan7faPf+LGCOL8u4kmhYuCf2R5f1qsWedAFJEzoQZ6DV9DbFrTw7AKjRNRl0mbrH7XvodNh7DjBm3a1dhUbxSlr4E0IYECMe5YpYt0BtZlEZYsW2jKqOnOcJ9F9cit2nzBXnG9dlVgLVyV6BVFzrarn8OAcgCDpE7kqdbr91aaxXf22MFaWfSUBSS2VG2tZbv0yCLjtLIslHq7LFO0luCebWk5Nltbt41guZu2JdwyvuSIm0YzzoRajaed61vsJDAU2PBUXQqRNcjvr6CZiLuGQGIUlHiyoBgzuO9js907MpXRmZ7l2teI1iKxGYmr8dgKwPUwDAMwHUq6/zuQ5gq29t5wQR0OEuSzleoFfTmkLwJ0QoFrwtQw7UF9R9IQ6dlhuJ7GQMOcr9etViYWBlPIoUbCV6H354BqpSiEkgE8pFWSKO4uOAeawxg2YMQJ8FpiUiwEW0IbbBriMyrwkme3oELdhrjAa9B70LjuaEWEY3U6ELTHj5fv52mBahvWUwrMgPK+A3Mu4e8VonSnDynALHqi9QiNeA6mDthw+q3nCBZdGrM7NluDKR68ngxenQ7Ir8CCOc2SJVgD/TeDk6CztV85K9JVFarwc0fRUOMkwiDbQGYISR9uXCosxEc8z+VxyFXPXZw65wh1y5lsGTpYlqcTfF31D7/uOOuPdmGi4B80K+JeqVkm/MN+iANhzNJ3efKe3Zf9s3EDb/fI8abzdjSxu1m2LtNblC8mpOhB2TXnmYWkR48eDTNSAnWwJSXiFKJKbzkBWxAewbwi8mgBrOMcJ+DQluAZjD5KYNgG2H8GhCxOyhz0hw71arDijM/eg/UK0+VhQD+/DiTSaT5HKeqiThaCQcLeh10XP3QOhnaBseo+ERapUwt2Gy9jYT/1QbrnVeCOLNqos7jydlZUB61qRKyapAg77e74k33FkpDQNLx4VpfzEHSpanDyQzJ+cXL0A0bU/TNJnG+caBsmPgGYi7EGxh4A7emNaFrM8ir7NCyJrvECAdNQRarw5Ozoh2T4FvSh9T5+8WlI2jXPxbgxblXct0eqV7OpStAcClMXe3cztqrI2bmwhJGmxWmJB6ELWRPD35wby/zwEDbky4h8+bcNyxPG9m74peAwoDr1ixQkdebplC7qHv7ZDe5ra1eo5jkvlSqsgMMLiufuvfrwFzBlnsNlmqMGQEkbEeHkbzLXhNwCOeMIqyNwrAtan8aB/M7nPKd4ZA+CnizoNZ66FuB7oyNertZ4hCO8EEc6wWJrWMl3NoTdBu69jr7Vec6uGYhE8KRA1YDwpJYUFwuE9cj/gSR0XMO0yMCwqXDJwtyR7Z5Yw/+JyvrF8NVoTEanp0MwTabDHUo6xGBMkwJG4/PhZIpHYWe+gKf2dNQRqxuriupDs6h1+NVmNBWZ2zj48+aBNhQREasyxCHiUubsWsalrKPW1pRvvh9Ohub81I6vWyeu35HnTkdAU0TCeveArFXmQKQ2t9NA9LooOToce2H84vUxBjXNms6HUw1h/3nkO9rtH3rdQef8th8olU1sFH4rWAjAWqTgI87SP/43x6wYw0Zt6P4sjHkRpVwEiS+0tGZ8nVe17Kn4kmoPu2nn7yNblKPghKDw54shqfGNm6LfTZQIf6nxe2InE+E7crhLpIARprrW4UOakSWo7RLjiGosi2clgl3BYQmYuqHlPenWGkrbsbJXocb+zvEGfXvjhJyslIe+QURd+J8TZEpqnU3I6NX4bDKU8ssIhh8HJxfDcxI+F0JF/esEfi4yPyfjYQt5OJkmkZdAGnS/Ktit2Nm9BISSk2msRovg0SMVYGZnztY4ruhMG8Kz2lt81i36zwHlx8MJRv4NeOR4eH5ETkanoyk59IgFC6udTiw0CEbG3dmARBWSQPvL6Kx8vww8Sw+uRJxTNdCcdIUJQrrQxkxw5TG+fPQuIFGr6pNDCyp9kiCi0+2JP2+s3Dpe7IuIiE1aZVWEOqFmT0IX5O3Jk4mIJ0snskR1x8MLuxlhzwn80l0JYb3kyFH4JpHIyPyCvQNftTByXxkhPT1ORB7bdkfPSkY01oc8CfAsysr2kQEuR2dEMjW0R67v8ajArfOM5hyZ9Uxu5qUVHcWwHnpltmr2BAd9o3voHCAD5177wgpcjGCK6I+M+Yko1c7Bm9aZDpEaw0YepCar9XUmAt2Nqb3wyo0DiDZHaIVaxpEcm3+iuhJmJfUu1xViE4zuQpPJt8Z0IRloE3B60QHLhSsA3fOSzoRN03QE6lwu5YfpbEsYOrQzwOxjGBQ8+HLY7aL7YDeD16+/+WanFzEx03LVXbgp5qzEVvYOLaF2Xa8yFjqldWN1KtvXzBG72sleum7aJ4HMZk2WNF/TrBEqwTqW1OgHqhXE1cJIA1umvE2KGw+tPD8coZUt1zfkaGXo1khoBzLai9i9BpjkabdrY2z7SurFtg+M/VRwIfzUBdh7AiiZ0WdMQBHsyzHaAOtXh0qBY6qFyC4NSsADx7Cmatxd/j5oK2zAoktYmqgbqPJ19GWMYvcWM/2+Kf94J4OATWyPBhsD+FiuaJUi4wNqgM1rmWCtr5HXqYzeutRqqTwX1cQRUrhKX5/NR8xRi+Mc9+AvtntrU+RBdqNR/pHBhmsKSrOuVstg1qGD59h50uMTHfo+Z0/jxVbuTVzZpoOuM4bPFktQW0d2/oZeyXcaKjS/9EKlPdggkme6j1cYoSbwc77h+sZ4G2bu+GzLFuWP1IrmDO8tcIywEW6upKBjCIwAPPqRi7ssCnNNeljUIKi1wcotYbXRym3H0rebmib1x+tJPcyr8lHKFu5qBZl26BJLuDvxKGngyb+t2JSL14UrNlHGuaZeCwTJ88Z0VYfKoXXIpI3XejGRPWjb4GptdUuXecCQWeT9ZrKP6Oqm8DS77lR8/oAx5pDgaVLf09scfQJO0lnVOJlsTI53quI5Q/EP3cpQDQxj0Hki7HHMCvpMzeMHBn/eWK6lfnmRvsP4OOzI8hpsRTKHbcGlgqaRhyAyLOZRpmL4tYh2i/Ndjgk61Q2AmOH/GuSoPqkFqYbp57xM7zwo95/EWLNIqnq3pgUqI7Hw0ENIUrSL6kiRQHuyNn+ruKNhbYw7yt4i7GhuH4DwbyuIXVyOP3WpQY5V87aVN/gJ8elPiJY+d90v73o8Gk2NKnxU3/WZhle3Y4FK8IoDJePrNvZ2k8PrbrVKc7NdVb3tG/KPcr68LtBdcAeqcwt7aLsFHfI5CZIA/rZXglXxu4/1ftgkqk9/C/oBb7X5nRKocKe/7B12r6BUQucMHMvchockOqCXhz1Fddk8PITahkhsyhvR2RE2jrxEyaNwvId2B7tWSmvU3dp4uacSdmnY8qrgtfEfeCMnCqYMD+xZovO0ZFqJR65YOf+NqG3jLFCYqfq6aTymSzafyr7F/UsoCgHrfWsfgLzWi0V61w/iarkCWoFlsYqpHFG0b5Xoa+PGgum1JSPFob3n6FXVfihSYH+xRWaDlHxbYkyh3+1s7rzI1uWNR9yCuF6U9/nMwKDPzN2m0EzzhQZeSjDHp2gfsOLRgM4324iGeJ1naf4+XKYlJq20yfhhO18LkpbpoWRJ0wDxShU7hgY2Sc9O79sm5/0xokbercCpzPvv4+gxr1ME8edztvwSJFLjxDLt03WS0UyBTVEZxWhxrIBrFzrTWxy+q+522qUqchIvdVmdS+l1rH0Zw4MnP1/9/uU/AoHHWN+R8HSm+X2YljIfEShNp8lL/KnB8f54nJYLTEVj5rKBDHSLw5s+6W7anUXdTKwcoNFz1M86q7GjAtuz1LEvN4UT4rqpaLbLyx+JfNk5netrAODpGNhUJKx0Yx5mlsZlJnVTqsV5gq6si02NdGqsdq7aeEM3fugvcg2s+o6ACd3Iy0n34tYFsou8G1B7dLQh1W0A++2w2DzVHGKDat0RsENf23Fk8lx7HgCkT9Nv4czxmLcALuIo1lAtR8Vu/qwVmdkH52CymwnwiJqVmHqhfWM01ZWEaGDYoFDDZl+9wPv7HcwIZGFnA1ab9ngbstcS4QiUwbh/x82dsL4WKG7euF/G1f33PeTUd37/f565dedqh/DdIT23qAvV2qS0twWtKdKJ6qbAJE43lYCDMJ1bDTi37rcgoVuVYM2AlycTMpVJ2br4EqEu4R+SnObKyrGlU0vv+cSRhzU/0xc9gYqRVJ9ZDLcPpe4prm7AEgA3FyY0ENZCC7+TUmoW0TlVKLhAYqZSpJVEfASFlvseu0s63HZ/bBNxflq0qKZVmMZWaLbajWyC8J5M7mY1z8LMBfCd/LY/721foyVOtyFLi+W4dVuio0myfUHcXK7ayrdeao62qio9paN195pt65a2bRdZ5t6+2Da8IwiaosTY0zKIWCdX4x2MRt75/mF+yQPlA8P2WuxaIUznqqcK20S77ozKdq1ZxAmAif2Y+5utWesEj1ZVM2aiQvhohbvsZnJMfBhQyzVrE4DpHoF9LHDZvfIdhzxgWJMAsm1YdZV4/1H9AmLrHI1ryp84lStkN06o79MCDYcq8t035/wa0339YHIyyr5+iNx5+wpDfjbbBmLfXbeO+5f9SwGcvl6JzovCDCb8yjZXFnvq2/TSyW1eF9TJhvoK09ddwcMZqEUxj3s54pjdrll2KwLAItjAr5XVufzjnyXBpQpIMA1B+bH3yo9p34Z4gFAQuQt7ioQ3o+n3FnDCxAEWD70boC+8x5YKqY04135rOL62bWabZdt0NLeNNdtOy5VcxSdbs+W214GvLbW0db7cq4vy2L1vrt/xkxOR7iPKGxf+t0xh6Zi8qWFwTY2vKoDBET/gqGpy9iYZX5y+GE6ATM6AH0n4ejCZjqb4GTK8SW/tn4Z6V8zdyGp7+SIfr4F2VdYEWBbvjuwbPFkBYuwqP3DAS4EW7zBSiimXhcsUxA0GVq4SEs0nM7wDul8mmLW/TaCC/Q6pN+L9+NuYnv5YqUqX++Sgeqn9w1bvWm02N8LYNdpBlKmSviMIhzuX9A6/1qUTqb3a1koJIZe1BhACNcUTrlwKnkZcQIfnpBSGBh1PfbziqzDQS27GNkSTy8BiiuBKu2EZp/MytEdp8k7QMtDVOmK6WoF4UZ3bcVJjixjd0PgSyJ464qvuViVxhB+VFBgsGIajQZhSkXUmdjcSj0Y54IsKdLGcoDP371UUvJizHADerSPy+HHkEUQysKVtSP5AMabtJ0sUbB3BWDgeIZh7hKAr3NV3T2zm3CJ4NkkaKYeMWPp3Ch65bePjszc13h+KhO0S29IK+6oAj2jfuofWnv9lsNcq3P7yTLPDRjnu+xgVwqs4WINmf9cHapdpvo0DBj++CutPBmlyF99MC12i/Pyw88VTzJNyS59CqZ++7J/44pByoofjYwkZA9LNNxsy9heLsDm9w4WI4hpiyeD9Q+8MO7jf/fDQVgaUzGfE0savBUlrbYOts1mj1sMosfunalShaC5r1djWplcH/wdQSwMEFAAAAAgAAAAhAJ494MUxAgAAqQMAABMAAABzcmMvc2NvbnRvL3J1dGFzLnB5ZVPBjpswEL0j8Q8j9gCREO1G6iXSHmjCppHSsEvoVtqqQo6ZEGuJjWyg2/5Tv6I/1jEhFav6hN94np/fPDzPy7qWGSixhkarn8hbFUGuSgVcSWhYe6rFYQE1A850gy0D77OQqAUDaoKStcp40AqUCKt0D2gaxoUyked5rnPU6nwlAXFulG7hgbauM26UcR3XyeLNM9wNlYBaUfZCKxlV2Ab+fpnu8rSwR/zwcqQojqLGophFGo2qewxmUcM0ytZ8m3+fzVyHeu43a6IcmN+BT485isp3nVWcxxOc9DPfCvhK4FAjULMfhH3M0t1zMoEPpOkXUmW/2T4l2aRiBInQVFmn29UEr1RdErqMl5+mRJzxk+XZpuv9REutKmO1JA9plk8LGq1Xtpbs83iVFvdfkl2e7CeUaFpWquLYkQloIm56H+AGekE4TfJ8EJKV7B/BapP91zw0dBKY5ifRK6A7gdiUlgqCSrSn7hBCrTiryeB4m2RvjLwA0dm+13VuIOlFiZJTTGy04p5JjnC7AJqXgppi1mkWwa6TnIFBig3X4oDATn9+R+TwMPSRfpwtXfoU75bJLcFjffDGdLWVbwomO1ajKebv5x9IxePjw/BJ0cnetAyZLZQWFZlCDb4Fr6cjzfTwBNcp8QjMYEVKdVEKbf8NTdEOZgvXAVpHcqgBISGg/IRwCUwIl3iEYMMQwjD8EOysQxhneyWwq4nOL0QejAG+y3WHIeArTa5QL8OW/P4LUEsBAhQAFAAAAAgAAAAhAJnbiMxOAAAAUAAAABYAAAAAAAAAAAAAAIABAAAAAHNyYy9zY29udG8vX19pbml0X18ucHlQSwECFAAUAAAACAAAACEAduobeOQLAADLJQAAGQAAAAAAAAAAAAAAgAGCAAAAc3JjL3Njb250by9jb25zdWx0YV92Mi5weVBLAQIUABQAAAAIAAAAIQBEhvvK0wQAAOMLAAAjAAAAAAAAAAAAAACAAZ0MAABzcmMvc2NvbnRvL2N1YWRlcm5vX29wZXJhdGl2b192Mi5weVBLAQIUABQAAAAIAAAAIQA06VOqUwsAAGEiAAAbAAAAAAAAAAAAAACAAbERAABzcmMvc2NvbnRvL2RlY2lzaW9uZXNfdjIucHlQSwECFAAUAAAACAAAACEA8QQYMzgXAACQWAAAFgAAAAAAAAAAAAAAgAE9HQAAc3JjL3Njb250by9pbnRyYWRheS5weVBLAQIUABQAAAAIAAAAIQCePeDFMQIAAKkDAAATAAAAAAAAAAAAAACAAak0AABzcmMvc2NvbnRvL3J1dGFzLnB5UEsFBgAAAAAGAAYAqgEAAAs3AAAAAA==')
sha_codigo = '6fc32491a9999a307fa2e892111062422850b48ade2be7004fcff61e3ccfc443'
assert hashlib.sha256(bundle).hexdigest() == sha_codigo
raiz = Path('sconto_v2_operativo_' + sha_codigo[:12]).resolve()
with zipfile.ZipFile(io.BytesIO(bundle)) as archivo:
    for nombre in archivo.namelist():
        if not re.fullmatch(r'src/sconto/[a-z_0-9]+\.py', nombre):
            raise ValueError('Ruta de código inesperada')
        destino = raiz / nombre
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(archivo.read(nombre))
sys.path.insert(0, str(raiz / 'src'))
import sconto
from sconto.cuaderno_operativo_v2 import reproducir
assert Path(sconto.__file__).is_relative_to(raiz), 'Reinicia el kernel para cargar la versión fijada'

resultado = reproducir(INTRADIA, CATALOGO, MOMENTO, productos=PRODUCTOS, privadas=PRIVADAS)
resultado['sha256_codigo'] = sha_codigo
SALIDA.write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print('Regla', resultado['regla'], '· ofertas visibles', resultado['numero_ofertas'],
      '· con historia previa suficiente', resultado['numero_ofertas_con_historia_previa'],
      '· evaluables', resultado['numero_ofertas_evaluables'])
display(JSON(resultado))